# HW5 Cats vs Dogs：MLP vs CNN vs CNN+Aug

資料集：TFDS `cats_vs_dogs`（二分類）

作業說明：
- ** 1 組 MLP / 1 組 CNN / 1 組 CNN+Aug**
- Epoch、影像尺寸、模型深度全面縮小
- **保證能比較三者優劣**


In [1]:
import sys, time, numpy as np, tensorflow as tf, tensorflow_datasets as tfds
import matplotlib.pyplot as plt

print("TF:", tf.__version__)
print("TFDS:", tfds.__version__)

SEED = 1
tf.random.set_seed(SEED)
np.random.seed(SEED)


TF: 2.15.1
TFDS: 4.9.9


## 1) 載入資料

In [2]:
(ds_all,), info = tfds.load(
    "cats_vs_dogs",
    split=["train"],
    as_supervised=True,
    with_info=True
)

NUM = 4000  # 只用 4000 張就好
ds_all = ds_all.shuffle(10000, seed=SEED, reshuffle_each_iteration=False).take(NUM)

ds_train = ds_all.take(3000)
ds_val   = ds_all.skip(3000).take(500)
ds_test  = ds_all.skip(3500).take(500)


2026-01-09 22:15:17.851556: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2026-01-09 22:15:17.851699: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-01-09 22:15:17.851704: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-01-09 22:15:17.851920: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-09 22:15:17.852075: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## 2) 前處理

In [3]:
IMG_SIZE = (96, 96)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(img, label):
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.float32)

def prep(ds, shuffle=False):
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(512, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train_p = prep(ds_train, shuffle=True)
ds_val_p   = prep(ds_val)
ds_test_p  = prep(ds_test)


## 3) MLP

In [4]:
from tensorflow.keras import layers, models

def build_mlp():
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE,3)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.legacy.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

mlp = build_mlp()
mlp.fit(ds_train_p, validation_data=ds_val_p, epochs=5, verbose=2)
mlp_test_acc = mlp.evaluate(ds_test_p, verbose=0)[1]
print("MLP Test Acc:", mlp_test_acc)


Epoch 1/5


2026-01-09 22:15:18.411903: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


94/94 - 7s - loss: 5.9595 - accuracy: 0.5297 - val_loss: 2.4684 - val_accuracy: 0.5660 - 7s/epoch - 71ms/step
Epoch 2/5
94/94 - 5s - loss: 1.6785 - accuracy: 0.5607 - val_loss: 2.3998 - val_accuracy: 0.5120 - 5s/epoch - 50ms/step
Epoch 3/5
94/94 - 4s - loss: 1.1375 - accuracy: 0.5837 - val_loss: 0.9920 - val_accuracy: 0.5580 - 4s/epoch - 47ms/step
Epoch 4/5
94/94 - 5s - loss: 0.8664 - accuracy: 0.5863 - val_loss: 1.0504 - val_accuracy: 0.5440 - 5s/epoch - 49ms/step
Epoch 5/5
94/94 - 4s - loss: 1.0833 - accuracy: 0.5743 - val_loss: 1.0269 - val_accuracy: 0.5620 - 4s/epoch - 47ms/step
MLP Test Acc: 0.5260000228881836


## 4) CNN

In [5]:
def build_cnn(use_aug=False):
    aug = []
    if use_aug:
        aug = [
            layers.RandomFlip("horizontal", seed=SEED),
            layers.RandomRotation(0.05, seed=SEED)
        ]

    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE,3)),
        *aug,
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.legacy.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn = build_cnn(use_aug=False)
cnn.fit(ds_train_p, validation_data=ds_val_p, epochs=5, verbose=2)
cnn_test_acc = cnn.evaluate(ds_test_p, verbose=0)[1]
print("CNN Test Acc:", cnn_test_acc)


Epoch 1/5
94/94 - 6s - loss: 0.7780 - accuracy: 0.5373 - val_loss: 0.6752 - val_accuracy: 0.5780 - 6s/epoch - 69ms/step
Epoch 2/5
94/94 - 6s - loss: 0.6288 - accuracy: 0.6443 - val_loss: 0.6988 - val_accuracy: 0.6080 - 6s/epoch - 60ms/step
Epoch 3/5
94/94 - 5s - loss: 0.5654 - accuracy: 0.7130 - val_loss: 0.6480 - val_accuracy: 0.6340 - 5s/epoch - 57ms/step
Epoch 4/5
94/94 - 5s - loss: 0.4977 - accuracy: 0.7630 - val_loss: 0.6223 - val_accuracy: 0.6940 - 5s/epoch - 58ms/step
Epoch 5/5
94/94 - 6s - loss: 0.4289 - accuracy: 0.8157 - val_loss: 0.6472 - val_accuracy: 0.6720 - 6s/epoch - 59ms/step
CNN Test Acc: 0.7419999837875366


## 5) CNN + 影像擴增

In [6]:
cnn_aug = build_cnn(use_aug=True)
cnn_aug.fit(ds_train_p, validation_data=ds_val_p, epochs=5, verbose=2)
cnn_aug_test_acc = cnn_aug.evaluate(ds_test_p, verbose=0)[1]
print("CNN+Aug Test Acc:", cnn_aug_test_acc)


Epoch 1/5


2026-01-09 22:16:16.963869: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:18.777604: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:18.916765: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:18.940948: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:18.962243: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:18.988819: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

94/94 - 7s - loss: 0.8377 - accuracy: 0.5317 - val_loss: 0.6591 - val_accuracy: 0.6200 - 7s/epoch - 74ms/step
Epoch 2/5


2026-01-09 22:16:23.617099: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:25.310045: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:25.348744: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:25.381778: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:25.417022: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:25.446261: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

94/94 - 6s - loss: 0.6620 - accuracy: 0.6280 - val_loss: 0.6762 - val_accuracy: 0.5800 - 6s/epoch - 66ms/step
Epoch 3/5


...

94/94 - 6s - loss: 0.6218 - accuracy: 0.6590 - val_loss: 0.6452 - val_accuracy: 0.6520 - 6s/epoch - 64ms/step
Epoch 4/5


2026-01-09 22:16:35.895550: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:37.491351: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:37.532434: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:37.563732: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:37.596988: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:37.627499: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

94/94 - 6s - loss: 0.5812 - accuracy: 0.6987 - val_loss: 0.5888 - val_accuracy: 0.7000 - 6s/epoch - 66ms/step
Epoch 5/5


2026-01-09 22:16:42.087394: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:43.653504: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:43.685110: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:43.716290: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:43.766115: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-01-09 22:16:43.789302: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

94/94 - 6s - loss: 0.5466 - accuracy: 0.7283 - val_loss: 0.5838 - val_accuracy: 0.7400 - 6s/epoch - 65ms/step
CNN+Aug Test Acc: 0.7400000095367432


## 6) 三者比較

In [7]:
print("===== Summary =====")
print("MLP Test Acc    :", mlp_test_acc)
print("CNN Test Acc    :", cnn_test_acc)
print("CNN+Aug Test Acc:", cnn_aug_test_acc)

best = max(
    [("MLP", mlp_test_acc), ("CNN", cnn_test_acc), ("CNN+Aug", cnn_aug_test_acc)],
    key=lambda x: x[1]
)
print(f"Best model: {best[0]} (test_acc={best[1]:.4f})")


===== Summary =====
MLP Test Acc    : 0.5260000228881836
CNN Test Acc    : 0.7419999837875366
CNN+Aug Test Acc: 0.7400000095367432
Best model: CNN (test_acc=0.7420)
